# 训练崩溃分析

本节通过真实的训练崩溃案例，介绍如何诊断和修复 RL 训练中的策略崩塌问题。

---

## 1. 案例背景

使用同一个 SFT 模型（Qwen3-1.7B-Wordle-SFT），在不同超参数下进行训练。其中一个 run 在 75 步后崩溃，另一个稳定训练 155 步。

---

## 2. 正常训练 vs 崩溃训练

### 2.1 指标对比

```text
正常训练 (稳定 run)

step   entropy  correct  response_length  reward
1      0.57     15%      2078             0.74
50     0.24     40%      1737             1.04
100    0.15     50%      1664             1.16
155    0.22     70%      1255             1.20
-> 稳定: entropy 缓慢下降到 0.22, correct 提升到 70%, response_length 稳定

崩溃训练 (同一模型, 不同 run)

step   entropy  correct  response_length  reward
1      0.56     15%      1755             0.76
50     0.24     40%      1737             1.04
75     0.15     45%      1650             1.09
100    0.001    0%       3232             0.20
150    0.0003   0%       3232             0.20
-> 崩溃: entropy 跌到 0, correct 归零, response_length 全部打满
```

### 2.2 关键差异

| 指标 | 正常 (step 155) | 崩溃 (step 150) |
|------|----------------|-----------------|
| entropy | 0.22 | **0.0003** |
| correct | 70% | **0%** |
| response_length | 1255 | **3232 (全部打满)** |
| num_turns | 5.0 | **6.0 (全部打满)** |
| kl_loss | 0.13 | 0.35 |
| reward | 1.20 | **0.20** (仅 format) |

---

## 3. 崩溃诊断流程

当训练出现异常时，按以下流程诊断：

```text
崩溃诊断流程

Step 1: 检查 entropy
  - 缓慢下降 -> 正常
  - 突然跌到 0 -> 策略崩塌
  - 持续飙升 -> entropy bonus 过强

Step 2: 检查 response_length
  - 稳定 (约 1700) -> 正常
  - 全部打满 (3232) -> 输出退化

Step 3: 检查 correct 率
  - 持续提升 -> 正常
  - 突然归零 -> 策略崩溃

Step 4: 检查 kl_loss
  - 缓慢上升 -> 正常
  - 飙升过快 -> 策略漂移过大

Step 5: 检查 grad_norm
  - 稳定 (0.4-1.0) -> 正常
  - 突然飙升 -> 梯度爆炸
```

---

## 4. 崩溃原因分析

策略崩塌的根因是 **三种力失衡**——策略梯度（pg_loss）压过了 KL 正则和 entropy bonus：

```text
三种力失衡分析

正常训练:
  pg_loss (降低 entropy) <-> kl_loss (限制漂移) + entropy_bonus (推高 entropy)
  三者平衡 -> 稳定 run 的 entropy 缓慢下降到 0.22

崩溃训练:
  pg_loss 过强 -> entropy 快速下降 -> 探索能力丧失
  -> 模型输出退化为固定模式 -> correct 归零
  -> response_length 全部打满 (固定输出填满 token)

根因: entropy_coeff=0 (无 entropy bonus) + kl_loss_coef=0.001 (太弱)
-> pg_loss 无约束地降低 entropy -> 崩塌
```

---

## 5. 修复方案

根据崩溃原因，修复策略是增强稳定性机制：

| 方案 | 操作 | 适用场景 |
|------|------|----------|
| 加 entropy bonus | `entropy_coeff=0.002` | entropy 下降过快 |
| 提高 KL coef | `kl_loss_coef=0.01` | 策略漂移过大 |
| 降低学习率 | `ACTOR_LR=5e-7` | 梯度更新过快 |
| 组合使用 | 以上多个 | 严重崩塌 |

> **注意**：entropy_coeff 不宜过大。实测过大的系数会让 entropy 快速飙升、模型过度探索而不收敛。本 recipe 使用已完整验证的 `0.002` 作为起点；调整后应先做短程实验，再决定是否完成 155 步训练。

---

## 6. 训练监控最佳实践

1. **前 50 步密切监控**：策略崩塌通常发生在训练早期
2. **设置 checkpoint 频率**：`save_freq=25`，崩溃时可回退
3. **观察验证指标**：每 5 步的 val 指标比训练指标更可靠
4. **同时看多个指标**：单一指标不能判断训练健康度
5. **准备回退方案**：保留稳定 run 的 checkpoint

---

## 课后练习

1. （判断题）策略崩塌的典型表现是 entropy 突然跌到 0、correct 归零、response_length 全部打满。

2. （判断题）策略崩塌的根因是 KL 正则过强，限制了模型学习新策略。

3. （判断题）entropy_coeff=0.01 比 0.002 更安全，因为更强的探索保证能更好地防止崩塌。

4. （单选题）崩溃训练中 reward 降到 0.20，这个 0.20 来自哪里？
    A. correct_answer
    B. partial_answer
    C. format_reward (0.2 * 1.0)
    D. length_bonus

5. （单选题）以下哪个是修复策略崩塌的有效方案？
    A. 降低 entropy_coeff
    B. 加 entropy_coeff=0.002
    C. 提高学习率
    D. 减小 batch size

6. （多选题）崩溃诊断流程中需要检查哪些指标？
    A. entropy
    B. response_length
    C. correct 率
    D. grad_norm

7. （多选题）以下哪些是训练监控的最佳实践？
    A. 前 50 步密切监控
    B. 设置 checkpoint 频率
    C. 只看 reward 一个指标
    D. 同时看多个指标

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/04_tuning_and_troubleshooting/answer/04.03_answer.txt